In [35]:
import os
import numpy as np
import pandas as pd
from scipy import sparse
import pickle
import dataframe_image as dfi
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. Ucitavamo podatke

In [36]:
data_dir = "preprocessed_data"
labels = np.load(os.path.join(data_dir, "labels.npy"))
X_stripped_count = sparse.load_npz(os.path.join(data_dir, "X_stripped_count.npz"))
stripped_df = pd.read_pickle(os.path.join(data_dir, "stripped_df.pkl"))

# 2. Generisanje cestih skupova stavki i pravila pridruzivanja
Koristimo fpgrowth algoritam zato sto je efikasniji na nasem skupu podataka od apriori. Koristimo min_support=10 da bismo izdvojili skupove koji se pojavljuju dovoljno cesto.
Nakon odredjivanja pravila pridruzivanja delimo ih na:
* Opsta
* Pravila kod kojih je ishod odbijena reklama
* Pravila kod kojih je ishod odobrena reklama

In [37]:
frequent_itemsets = fpgrowth(
    stripped_df,
    min_support=0.10,
    use_colnames=True,
)
print(f"\nFrequent itemsets found: {len(frequent_itemsets)}")

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.6
)
rules = rules.sort_values('lift', ascending=False)
print(f"Rules generated: {len(rules)}")

rejected_rules = rules[rules['consequents'].apply(lambda x: 'label_rejected' in x)].sort_values('lift', ascending=False)
print(f"Rejected rules: {len(rejected_rules)}")

accepted_rules = rules[rules['consequents'].apply(lambda x: 'label_accepted' in x)].sort_values('lift', ascending=False)
print(f"accepted Rules: {len(accepted_rules)}")


Frequent itemsets found: 1802
Rules generated: 703
Rejected rules: 22
accepted Rules: 145


# 3. Vizualizacija
Pravimo funkciju koja kreira stilizovanu tabelu koristeci izdvojena pravila i pozivamo je nad datim skupovima.

In [38]:
def show_top_rules(rules_df, top_n=20, color_by='lift'):
    display_df = rules_df.copy()
    display_df['antecedents'] = display_df['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    display_df['consequents'] = display_df['consequents'].apply(lambda x: ', '.join(sorted(x)))
    display_df[['support', 'confidence', 'lift']] = display_df[['support', 'confidence', 'lift']].round(3)
    return (display_df[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
            .head(top_n)
            .style
            .background_gradient(subset=['lift'], cmap='Reds')
            .background_gradient(subset=['confidence'], cmap='Blues')
            .background_gradient(subset=['support'], cmap='Greens')
            .set_properties(**{'text-align': 'left'})
            .hide(axis='index')
           )

In [39]:
show_top_rules(rules, top_n=20)

antecedents,consequents,support,confidence,lift
code,zip,0.107000,0.682000,5.266000
zip,code,0.107000,0.827000,5.266000
dog,cat,0.102000,0.621000,5.080000
cat,dog,0.102000,0.837000,5.080000
symptom,"cause, treatment",0.105000,0.635000,4.398000
"cause, treatment",symptom,0.105000,0.726000,4.398000
"cause, treatment","disease, label_rejected",0.104000,0.720000,4.384000
"disease, label_rejected","cause, treatment",0.104000,0.632000,4.384000
"disease, treatment","cause, label_rejected",0.104000,0.640000,4.253000
"cause, label_rejected","disease, treatment",0.104000,0.690000,4.253000


In [40]:
show_top_rules(accepted_rules, top_n=20) 

antecedents,consequents,support,confidence,lift
dog,"label_accepted, pet",0.115000,0.701000,3.965000
photo,"label_accepted, video",0.110000,0.606000,3.058000
horse,label_accepted,0.126000,0.951000,1.717000
"animal, video",label_accepted,0.106000,0.928000,1.675000
"animal, pet",label_accepted,0.114000,0.923000,1.666000
"dog, pet",label_accepted,0.115000,0.917000,1.656000
cat,label_accepted,0.112000,0.915000,1.651000
dog,label_accepted,0.150000,0.911000,1.644000
"animal, care",label_accepted,0.105000,0.910000,1.642000
animal,label_accepted,0.179000,0.889000,1.604000


In [41]:
show_top_rules(rejected_rules, top_n=20) 

antecedents,consequents,support,confidence,lift
"cause, treatment","disease, label_rejected",0.104000,0.720000,4.384000
"disease, treatment","cause, label_rejected",0.104000,0.640000,4.253000
"cause, disease","label_rejected, treatment",0.104000,0.760000,3.545000
symptom,"label_rejected, treatment",0.118000,0.714000,3.332000
cause,"label_rejected, treatment",0.125000,0.628000,2.928000
doctor,"label_rejected, treatment",0.107000,0.625000,2.917000
"doctor, treatment",label_rejected,0.107000,0.895000,2.007000
patient,label_rejected,0.121000,0.882000,1.978000
"cause, disease, treatment",label_rejected,0.104000,0.878000,1.969000
"cause, treatment",label_rejected,0.125000,0.866000,1.943000
